# MVP — Previsão de Cancelamento de Clientes (Churn) em Telecom

**Autor:** Mateus M. Matos  
**Pós-Graduação em Análise de Dados com IA**  
**Tipo de tarefa:** Classificação binária supervisionada

---

Este notebook é um **relatório técnico executável**: cada etapa traz o código, o resultado e a explicação da decisão tomada. Ele roda do início ao fim sem necessidade de upload, login ou token — os dados são carregados diretamente de uma URL pública.

## 1. Apresentação do problema

### O problema
Empresas de telecomunicações perdem receita quando clientes **cancelam o serviço** — fenômeno chamado de **churn**. Conquistar um cliente novo costuma custar bem mais caro do que reter um cliente atual, então identificar **quem está prestes a cancelar** permite agir antes (oferta, contato, desconto) e economizar dinheiro.

### Objetivo do modelo
Construir um modelo que, a partir das características de um cliente (tempo de casa, tipo de contrato, serviços contratados, forma de pagamento, etc.), **preveja se ele vai cancelar (`Yes`) ou permanecer (`No`)**.

### Variável-alvo
A coluna **`Churn`**, que assume dois valores: `Yes` (o cliente cancelou) ou `No` (permaneceu).

### Por que isso é um problema de Machine Learning?
Porque queremos **aprender um padrão a partir de exemplos históricos**: temos milhares de clientes com o desfecho já conhecido (cancelaram ou não) e suas características. O modelo aprende a relação entre as características e o desfecho para depois **prever o desfecho de clientes novos**, ainda sem rótulo. Como a variável-alvo é uma **categoria** (`Yes`/`No`), trata-se de um problema de **classificação** — e binária, por ter apenas duas classes.

### Premissas, hipóteses e restrições
- **Premissa:** os dados históricos são representativos do comportamento futuro dos clientes (o padrão de cancelamento não muda radicalmente de um período para o outro).
- **Hipótese:** características como **tipo de contrato**, **tempo de casa (`tenure`)** e **forma de pagamento** têm relação com a propensão ao cancelamento.
- **Restrição:** trabalhamos apenas com os atributos disponíveis no dataset; não há dados externos (ex.: reclamações, uso real do serviço) que poderiam enriquecer o modelo.
- **Foco do MVP:** o objetivo não é o melhor modelo possível, e sim uma solução **coerente, reprodutível e bem justificada** de ponta a ponta.

## 2. Configuração do ambiente e carregamento dos dados

Importamos as bibliotecas e fixamos uma **semente aleatória (`SEED`)**. Fixar a semente garante **reprodutibilidade**: toda vez que o notebook rodar, as divisões aleatórias e os modelos produzem o mesmo resultado — essencial para que o professor reproduza exatamente os nossos números.

In [ ]:
# Bibliotecas usadas no projeto
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Semente para reprodutibilidade (mesmos resultados a cada execução)
SEED = 42
np.random.seed(SEED)

# Ajustes visuais
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Bibliotecas carregadas com sucesso.")

Carregamos os dados **diretamente da URL pública** (versão *raw* do GitHub). Esse é o ponto crítico de reprodutibilidade: qualquer pessoa executa o notebook sem precisar baixar nada manualmente.

In [ ]:
# Dados carregados de uma URL pública -> sem upload, sem login, sem token
URL = "https://raw.githubusercontent.com/MateusMMatos/mvp-churn-telco/main/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)

print(f"Dataset carregado: {df.shape[0]} clientes (linhas) e {df.shape[1]} atributos (colunas).")
df.head()

## 3. Conhecendo os dados (análise exploratória)

Antes de modelar, precisamos **entender** os dados: quantas linhas e colunas temos, de que tipo é cada variável, se há valores ausentes e como a variável-alvo está distribuída. Essa etapa evita erros grosseiros mais adiante — lembrando o princípio *"garbage in, garbage out"*: modelo bom com dado ruim não existe.

### Sobre os dados: fonte, variáveis e limitações

- **Fonte:** dataset *Telco Customer Churn*, disponibilizado publicamente pela IBM como base de amostra. É um conjunto de dados **fictício** de uma operadora de telecom imaginária — criado para fins de ensino, sem representar clientes reais (portanto, sem qualquer questão de privacidade/confidencialidade).
- **Volume:** 7.043 registros (clientes) e 21 atributos (confirmados na célula acima).
- **Variável-alvo:** `Churn` — indica se o cliente cancelou (`Yes`) ou permaneceu (`No`).

**Dicionário das principais variáveis:**

| Variável | Significado |
|---|---|
| `customerID` | Identificador único do cliente (será removido — não tem poder preditivo) |
| `gender`, `SeniorCitizen`, `Partner`, `Dependents` | Perfil demográfico (sexo, se é idoso, se tem cônjuge/dependentes) |
| `tenure` | Nº de meses que a pessoa é cliente |
| `PhoneService`, `MultipleLines`, `InternetService` | Serviços de telefonia e internet contratados |
| `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | Serviços adicionais (Yes/No/sem internet) |
| `Contract` | Tipo de contrato (mês a mês, 1 ano, 2 anos) |
| `PaperlessBilling`, `PaymentMethod` | Fatura digital e forma de pagamento |
| `MonthlyCharges`, `TotalCharges` | Mensalidade e gasto total acumulado |
| **`Churn`** | **Alvo:** o cliente cancelou? (`Yes`/`No`) |

**Critérios usados para escolher esta base:**
- É um **problema de negócio real e comum** (retenção de clientes), com aplicação prática clara.
- É uma tarefa de **classificação binária**, alinhada ao que foi estudado.
- Tem **desbalanceamento** de classes, o que enriquece a discussão de métricas.
- Contém **variáveis mistas** (categóricas e numéricas), exercitando pré-processamento.
- É **pública** e carregável por URL, atendendo à exigência de reprodutibilidade.
- **Não foi utilizada** nas aulas da sprint.

**Limitações conhecidas do dataset:**
- Por ser **fictício**, tende a ser mais "comportado" que dados reais — os padrões podem ser mais nítidos do que na prática.
- É uma **fotografia estática**: não há informação temporal de *quando* cada cliente cancelou, nem histórico de uso.
- Faltam variáveis potencialmente ricas (reclamações, satisfação, chamados de suporte) que ajudariam a explicar o cancelamento.

### 3.1 Dimensões e tipos das variáveis

In [ ]:
# Quantidade de registros e atributos + tipo de cada coluna e contagem de não-nulos
print(f"O dataset tem {df.shape[0]} registros (clientes) e {df.shape[1]} atributos (colunas).\n")
df.info()

**Leitura do resultado:**
- São **7.043 clientes** e **21 colunas**. A maioria (18) é do tipo **texto** (`object`/`str`), como `gender`, `Contract`, `PaymentMethod`. O computador não treina com texto, então essas colunas precisarão ser **codificadas em números** na etapa de preparação.
- Apenas 3 colunas são numéricas: `SeniorCitizen`, `tenure` (meses de contrato) e `MonthlyCharges` (mensalidade).
- Um ponto de atenção com a qualidade dos dados: a coluna `TotalCharges` (gasto total do cliente) aparece como **texto**, quando deveria ser um **número**. Isso costuma indicar que há alguma sujeira escondida nela. Vamos investigar e corrigir na preparação dos dados.

### 3.2 Estatísticas descritivas das variáveis numéricas

In [ ]:
# Resumo estatístico das colunas numéricas (média, desvio, mínimo, máximo, quartis)
df.describe().round(2)

**Leitura do resultado:**
- `tenure` (tempo de casa) vai de **0 a 72 meses**, com média de ~32 — ou seja, há tanto clientes novatos quanto clientes de 6 anos.
- `MonthlyCharges` (mensalidade) vai de **R\$ 18,25 a R\$ 118,75**.
- Repare que o `describe()` só mostrou **3 colunas**. A `TotalCharges` não apareceu aqui, justamente porque o pandas a tratou como texto e não calcula estatística de texto. É a confirmação do problema que apontamos acima.

### 3.3 Distribuição da variável-alvo (`Churn`) — análise de desbalanceamento

In [ ]:
# Quantos clientes cancelaram (Yes) x permaneceram (No), em número e em %
contagem = df["Churn"].value_counts()
print(contagem)
print()
print((df["Churn"].value_counts(normalize=True) * 100).round(1))

# Gráfico de barras (matplotlib puro -> funciona em qualquer versão)
cores = {"No": "#2ecc71", "Yes": "#e74c3c"}
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(contagem.index, contagem.values,
       color=[cores.get(v, "#888888") for v in contagem.index])
ax.set_title("Distribuição do alvo (Churn)")
ax.set_xlabel("Cliente cancelou?")
ax.set_ylabel("Nº de clientes")
for i, v in enumerate(contagem.values):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.show()

**Leitura do resultado — este é um ponto central do projeto:**
- **73,5%** dos clientes permaneceram (`No`) e apenas **26,5%** cancelaram (`Yes`). As classes estão **desbalanceadas**.
- Por que isso importa tanto: um modelo "preguiçoso" que simplesmente chutasse *"ninguém cancela"* para todos os clientes já acertaria **73,5%** — sem aprender nada. Isso mostra por que a **acurácia sozinha engana** neste problema. Vamos precisar de métricas como **precisão, recall e F1-score**, que enxergam o acerto na classe minoritária (os que cancelam), que é justamente a que interessa ao negócio.
- Essa observação vai guiar tanto a escolha do **baseline** quanto a escolha das **métricas de avaliação** mais adiante.

### 3.4 Quais características puxam o cancelamento?

Agora investigamos **quem cancela mais**: para cada característica, calculamos a **taxa de cancelamento** (% de clientes daquele grupo que deram churn). Isso ajuda a entender o problema e antecipa quais variáveis o modelo provavelmente achará importantes.

A função abaixo calcula a taxa de churn de qualquer coluna categórica — criá-la evita repetir o mesmo código várias vezes (boa prática, §8).

In [ ]:
def taxa_churn(coluna):
    """Retorna a % de clientes que cancelaram (Churn='Yes') em cada categoria da coluna."""
    tab = (pd.crosstab(df[coluna], df["Churn"], normalize="index")["Yes"] * 100).round(1)
    return tab.sort_values(ascending=False)

# Taxa de cancelamento por tipo de contrato
tabela_contrato = taxa_churn("Contract")
print(tabela_contrato)

# Gráfico de barras (matplotlib puro -> funciona em qualquer versão)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(tabela_contrato.index, tabela_contrato.values, color="#c44e52")
ax.set_title("Taxa de cancelamento por tipo de contrato")
ax.set_xlabel("Tipo de contrato")
ax.set_ylabel("% que cancelou")
for i, v in enumerate(tabela_contrato.values):
    ax.text(i, v, f"{v}%", ha="center", va="bottom")
plt.show()

**Leitura:** o tipo de contrato é um dos fatores mais fortes. Clientes **mês a mês** cancelam **42,7%** das vezes, contra apenas **2,8%** dos clientes com contrato de **2 anos**. Faz sentido no mundo real: o contrato mensal não tem fidelidade nem multa, então o cliente sai quando quiser; já quem assinou 2 anos está preso (e provavelmente mais satisfeito).

In [ ]:
# Como o tempo de casa (tenure) se relaciona com o cancelamento
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, element="step",
             stat="density", common_norm=False,
             palette=["#2ecc71", "#e74c3c"], ax=ax)
ax.set_title("Distribuição do tempo de casa (tenure) por churn")
ax.set_xlabel("Meses como cliente (tenure)")
plt.show()

print("Tempo de casa médio:")
print(df.groupby("Churn")["tenure"].mean().round(1))

**Leitura:** o cancelamento se concentra nos **primeiros meses**. Quem cancela tem, em média, **18 meses** de casa; quem permanece, **38 meses**. O pico vermelho (churn) fica logo no início do eixo — ou seja, **cliente novo é cliente em risco**. Isso reforça a estratégia de negócio de investir na retenção logo nos primeiros meses.

---

**Síntese da exploração — o perfil de risco de cancelamento:**

| Sinal de risco | Grupo com mais churn |
|---|---|
| Contrato | Mês a mês (42,7%) |
| Pagamento | Cheque eletrônico (45,3%) |
| Internet | Fibra óptica (41,9%) |
| Tempo de casa | Clientes novos (~18 meses) |
| Mensalidade | Mais alta (~R\$ 74) |

Essas relações não só ajudam a entender o problema como indicam que **há sinal aprendível nos dados** — bom presságio para a modelagem. Nenhuma variável sozinha "explica tudo", então um modelo que combine várias delas tende a ir bem.

## 4. Preparação dos dados

Agora deixamos os dados prontos para a modelagem. Cada decisão abaixo é **justificada** — o objetivo não é só "aplicar transformações", mas explicar por que cada uma faz sentido. As transformações que dependem de aprender algo dos dados (como codificação e padronização) serão feitas **dentro de um pipeline ajustado só no treino**, para evitar **vazamento de dados** (data leakage) — assunto das próximas etapas.

### 4.1 Corrigindo a coluna `TotalCharges`

Na exploração, vimos que `TotalCharges` (gasto total do cliente) veio como **texto** em vez de número. Vamos investigar a causa antes de corrigir.

In [ ]:
# Investigar as linhas onde TotalCharges está em branco
brancos = df["TotalCharges"].astype(str).str.strip() == ""
print(f"Linhas com TotalCharges em branco: {brancos.sum()}")
df.loc[brancos, ["tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

**Diagnóstico:** as 11 linhas em branco são todas de clientes com **`tenure = 0`** (entraram agora e ainda não completaram o primeiro mês). Ou seja, o gasto acumulado delas é logicamente **zero** — não é um erro aleatório, é um vazio com significado.

**Decisão:** (1) converter `TotalCharges` para número e (2) preencher esses casos com **0**. Preencher com a média seria um erro grave — colocaríamos milhares de reais de gasto em quem nunca pagou uma fatura.

In [ ]:
# 1) Converter TotalCharges para número (os valores em branco viram NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Nulos após a conversão:", df["TotalCharges"].isna().sum())

# 2) Preencher os NaN com 0 -> são clientes com tenure=0 (ainda sem gasto acumulado)
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Tipo agora:", df["TotalCharges"].dtype)
print("Nulos restantes:", df["TotalCharges"].isna().sum())

**Resultado:** a coluna agora é numérica (`float64`), sem valores nulos, e passa a ser considerada nas análises numéricas.

> **Nota sobre vazamento de dados:** preencher com **0** aqui é seguro porque é uma **regra de negócio fixa** (se `tenure = 0`, então o gasto acumulado é 0), e não uma estatística aprendida dos dados. Se tivéssemos preenchido com a *média* de `TotalCharges`, essa média teria que ser calculada **só no treino** para não vazar informação do teste — cuidado que teremos nas próximas etapas.

### 4.2 Removendo o identificador e preparando a variável-alvo

In [ ]:
# 1) Remover o identificador do cliente (não tem poder preditivo)
df = df.drop(columns=["customerID"])

# 2) Converter a variável-alvo de texto para número: Yes -> 1 (cancelou), No -> 0 (permaneceu)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# 3) Separar atributos preditores (X) da variável-alvo (y)
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("Formato de X (preditores):", X.shape)
print("Formato de y (alvo):     ", y.shape)
print("\nDistribuição do alvo (0 = ficou, 1 = cancelou):")
print(y.value_counts())

**Decisões:**
- **`customerID`** foi **removido**: é apenas um código de identificação, sem poder preditivo. Mantê-lo poderia até atrapalhar (o modelo "decoraria" clientes em vez de aprender padrões).
- A variável-alvo **`Churn`** foi convertida de texto (`Yes`/`No`) para número (`1`/`0`), formato exigido pelos algoritmos. Convenção: **1 = cancelou** (a classe de interesse), **0 = permaneceu**.
- Separamos os dados em **`X`** (atributos preditores) e **`y`** (alvo).

### 4.3 Identificando variáveis numéricas e categóricas

Para tratar cada tipo de variável de forma adequada, separamos as colunas numéricas das categóricas.

In [ ]:
# Identificar automaticamente quais colunas são numéricas e quais são categóricas.
# Usamos "number" (numéricas) e o complemento (não-numéricas) -> funciona em qualquer
# versão do pandas, evitando problemas com o seletor de tipo texto.
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Variáveis numéricas ({len(num_cols)}): {num_cols}")
print(f"\nVariáveis categóricas ({len(cat_cols)}): {cat_cols}")

**Leitura:** temos **4 variáveis numéricas** (`SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges`) e **15 categóricas** (texto). Essa separação é importante porque cada tipo recebe um tratamento diferente na modelagem:
- **Categóricas** → precisam ser **codificadas** em números (o modelo não entende `"Yes"`, `"Fiber optic"`).
- **Numéricas** → serão **padronizadas** (colocadas na mesma escala), o que ajuda modelos sensíveis a escala.

Esses dois tratamentos serão montados num **pipeline** e ajustados **apenas no treino** (próximas etapas) — é assim que evitamos o vazamento de dados.

## 5. Divisão dos dados em treino e teste

Para avaliar honestamente se o modelo aprendeu, escondemos uma parte dos dados (**teste**) e treinamos apenas na outra (**treino**). O modelo só é avaliado em dados que **nunca viu** — assim medimos se ele generaliza para clientes novos, e não se apenas "decorou" os exemplos.

### 5.1 Separando treino (70%) e teste (30%)

In [ ]:
from sklearn.model_selection import train_test_split

# Divisão 70% treino / 30% teste, mantendo a proporção de churn nos dois grupos (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

print(f"Treino: {X_train.shape[0]} clientes | Teste: {X_test.shape[0]} clientes")
print(f"% de churn no treino: {y_train.mean() * 100:.1f}%")
print(f"% de churn no teste:  {y_test.mean() * 100:.1f}%")

**Leitura:** separamos **70% para treino** (4.930 clientes) e **30% para teste** (2.113 clientes). O parâmetro `stratify=y` garante que **a proporção de churn (26,5%) seja a mesma** nos dois conjuntos — importante num problema desbalanceado, para que o teste seja representativo. O `random_state=SEED` torna a divisão **reprodutível**.

### 5.2 Pré-processamento à prova de vazamento

Agora definimos como as variáveis serão transformadas. Em vez de aplicar as transformações "na mão" (o que arriscaria vazamento), montamos um **`ColumnTransformer`** — uma "receita" que será executada **dentro do pipeline de cada modelo**, ajustada apenas com os dados de treino.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Receita de pré-processamento: padroniza as numéricas e codifica (one-hot) as categóricas.
# ATENÇÃO: aqui só DEFINIMOS a receita. Ela será ajustada (fit) apenas no treino,
# dentro do pipeline de cada modelo (próxima etapa) -> sem vazamento de dados.
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
preprocessor

**Como isto evita o vazamento de dados:** o `preprocessor` acima é apenas a *receita* — ele ainda **não olhou nenhum dado**. Na próxima etapa, ele será encaixado dentro do pipeline de cada modelo e ajustado (`fit`) **somente com `X_train`**. Assim, a média e o desvio usados na padronização, e as categorias aprendidas na codificação, vêm **exclusivamente do treino** — o teste permanece "lacrado" até a hora de avaliar. É a forma correta e à prova de erros de tratar os dados.

**O que cada transformação faz:**
- **`StandardScaler`** (numéricas): coloca as variáveis na mesma escala (média 0, desvio 1). Sem isso, `TotalCharges` (milhares) dominaria `tenure` (dezenas) em modelos sensíveis a escala, como KNN.
- **`OneHotEncoder`** (categóricas): cria uma coluna 0/1 para cada categoria. Ex.: `Contract` vira 3 colunas (`Month-to-month`, `One year`, `Two year`). É por isso que os 19 atributos originais viraram 45 colunas.

## 6. Modelagem e avaliação

Seguindo o MVP, treinamos um **baseline** (referência ingênua) e **dois modelos candidatos**, comparando os resultados. Todos os modelos usam **pipelines** que aplicam o pré-processamento (ajustado só no treino) antes de treinar — garantindo ausência de vazamento.

**Métricas escolhidas — e por que:** como o problema é **desbalanceado**, a acurácia sozinha engana. Usaremos quatro métricas:

| Métrica | O que responde | Por que importa aqui |
|---|---|---|
| **Acurácia** | % de acertos totais | Referência, mas enganosa com desbalanceamento |
| **Precisão** | Dos previstos como "vai cancelar", quantos cancelaram? | Evita alarme falso (gastar retenção à toa) |
| **Recall** | Dos que realmente cancelaram, quantos o modelo pegou? | **A mais crítica**: cliente que escapa = receita perdida |
| **F1** | Equilíbrio entre precisão e recall | Resumo justo num cenário desbalanceado |

Para churn, **priorizamos o recall/F1 da classe "cancelou"**: deixar escapar quem ia cancelar (falso negativo) costuma custar mais caro do que abordar quem não ia sair (falso positivo).

Definimos abaixo uma função de avaliação reutilizável, para medir todos os modelos da mesma forma.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

# Lista que vai acumular as métricas de cada modelo (para comparar no final)
resultados = []

def avaliar_modelo(nome, modelo):
    """Avalia um modelo treinado no conjunto de TESTE e guarda suas métricas."""
    y_pred = modelo.predict(X_test)
    metricas = {
        "Modelo": nome,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
    }
    resultados.append(metricas)
    print(nome)
    for chave, valor in metricas.items():
        if chave != "Modelo":
            print(f"  {chave}: {valor:.3f}")
    return y_pred

### 6.1 Baseline — o modelo de referência

O baseline é a abordagem mais ingênua possível: **sempre prever a classe majoritária** ("não cancela"), ignorando os atributos. Ele serve de **régua**: qualquer modelo de verdade precisa superá-lo para justificar sua complexidade.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay

# Baseline: ignora os atributos e sempre prevê a classe majoritária ("não cancela")
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

y_pred_base = avaliar_modelo("Baseline (sempre 'não cancela')", baseline)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_base, display_labels=["Ficou", "Cancelou"], cmap="Blues"
)
plt.title("Matriz de confusão — Baseline")
plt.show()

**Leitura — o baseline é revelador:** ele acerta **73,5%** (a acurácia parece boa!), mas tem **recall = 0** e **F1 = 0**: não identifica **nenhum** cliente que cancelou. A matriz de confusão confirma — ele prevê "ficou" para todos, então "acerta" os 1.552 que ficaram e **deixa escapar os 561 que cancelaram**.

Isso deixa clara a lição central do projeto: **num problema desbalanceado, acurácia alta pode significar um modelo inútil**. Nossos próximos modelos precisam superar esse baseline **onde importa** — ou seja, no **recall/F1** da classe que cancela, não só na acurácia.

### 6.2 Modelo candidato 1 — Regressão Logística

A **Regressão Logística** é um modelo linear clássico para classificação binária: rápido, **interpretável** e um excelente ponto de partida. Ela estima a probabilidade de o cliente cancelar a partir de uma combinação das variáveis. Usamos um **pipeline** que junta o pré-processamento ao modelo — assim o `fit` aplica a padronização/codificação apenas no treino, sem vazamento.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Modelo 1: Regressão Logística (dentro de um pipeline com o pré-processamento)
pipe_lr = Pipeline([
    ("prep", preprocessor),
    ("modelo", LogisticRegression(max_iter=1000, random_state=SEED)),
])
pipe_lr.fit(X_train, y_train)

y_pred_lr = avaliar_modelo("Regressão Logística", pipe_lr)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr, display_labels=["Ficou", "Cancelou"], cmap="Blues"
)
plt.title("Matriz de confusão — Regressão Logística")
plt.show()

**Leitura:** a Regressão Logística deu um salto enorme em relação ao baseline: **recall subiu de 0 para ~0,56** (agora detecta mais da metade dos cancelamentos) e o **F1 foi a ~0,61**. A acurácia também subiu (~0,81). Ou seja: o modelo **aprendeu de verdade**, e não está só repetindo a maioria como o baseline.

### 6.3 Modelo candidato 2 — Random Forest

A **Random Forest** é um *conjunto* (ensemble) de várias árvores de decisão que "votam" juntas. Ela captura **relações não-lineares e interações** entre variáveis e costuma ser robusta em dados tabulares. É bem diferente da Regressão Logística — comparar dois modelos de famílias distintas dá uma visão mais rica.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Modelo 2: Random Forest (conjunto de árvores de decisão)
pipe_rf = Pipeline([
    ("prep", preprocessor),
    ("modelo", RandomForestClassifier(random_state=SEED, n_jobs=-1)),
])
pipe_rf.fit(X_train, y_train)

y_pred_rf = avaliar_modelo("Random Forest", pipe_rf)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf, display_labels=["Ficou", "Cancelou"], cmap="Blues"
)
plt.title("Matriz de confusão — Random Forest")
plt.show()

**Leitura:** a Random Forest também supera o baseline, mas neste caso ficou **um pouco atrás** da Regressão Logística. Modelos mais complexos nem sempre vencem — em dados tabulares com relações relativamente simples, um modelo linear bem ajustado costuma ser muito competitivo.

### 6.4 Comparação dos modelos

In [ ]:
# Tabela comparativa com as métricas de todos os modelos
df_resultados = pd.DataFrame(resultados).set_index("Modelo").round(3)
print(df_resultados)

# Gráfico comparativo
ax = df_resultados[["Acurácia", "Precisão", "Recall", "F1"]].plot(
    kind="bar", figsize=(9, 5)
)
ax.set_title("Comparação dos modelos")
ax.set_ylabel("Pontuação (0 a 1)")
ax.set_ylim(0, 1)
plt.xticks(rotation=12, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Leitura da comparação:**
- **Os dois modelos superam o baseline com folga** — ambos saíram de recall 0 para detectar mais da metade dos cancelamentos. Ou seja, eles **realmente aprenderam** padrões dos dados.
- **A Regressão Logística ficou à frente da Random Forest** em todas as métricas, inclusive no **F1** (0,61 vs 0,53) e no **recall** (0,56 vs 0,47). Um resultado comum em dados tabulares: o modelo mais **simples** venceu o mais complexo — o que reforça que "mais complexo" nem sempre é "melhor".
- Ainda assim, o **recall de ~56% significa que quase metade dos clientes que cancelam ainda escapa**. Há espaço para melhorar — é o que faremos na etapa de **otimização de hiperparâmetros**, tentando aumentar o recall sem destruir a precisão.

## 7. Otimização de hiperparâmetros

Até aqui usamos as configurações **padrão** dos modelos. Agora ajustamos os **hiperparâmetros** (os "botões" do modelo) da Regressão Logística — a melhor até o momento — buscando melhorar o desempenho na classe que interessa (quem cancela).

**O que ajustamos e por quê:**
- **`C`** — controla a **regularização** (o quanto o modelo evita ficar complexo demais). Testamos valores de 0,01 a 10.
- **`class_weight`** — permite dar **mais peso à classe minoritária**. Como o problema é desbalanceado, `"balanced"` pode ajudar o modelo a "prestar mais atenção" nos clientes que cancelam.

**Estratégia:** usamos **Grid Search** (testa todas as combinações) com **validação cruzada estratificada de 5 dobras**. A validação cruzada divide o treino em 5 partes e testa cada combinação em rodadas diferentes — assim escolhemos a melhor configuração **sem nunca tocar no conjunto de teste** (evitando vazamento). Otimizamos pelo **F1**, por equilibrar precisão e recall num cenário desbalanceado.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Espaço de busca: força da regularização (C) e balanceamento das classes
grade_parametros = {
    "modelo__C": [0.01, 0.1, 1, 10],
    "modelo__class_weight": [None, "balanced"],
}

# Validação cruzada estratificada (5 dobras) -> avalia cada combinação SÓ no treino
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid = GridSearchCV(pipe_lr, grade_parametros, scoring="f1", cv=cv, n_jobs=-1)
grid.fit(X_train, y_train)

print("Melhor combinação:", grid.best_params_)
print(f"Melhor F1 (média na validação cruzada): {grid.best_score_:.3f}")

**Leitura da busca:** a melhor combinação encontrada foi **`C = 0.01`** (regularização forte, modelo mais simples/conservador) com **`class_weight = "balanced"`** (dá mais peso à classe minoritária). A validação cruzada avaliou cada combinação usando **apenas os dados de treino** — o conjunto de teste continua intocado. Agora aplicamos o melhor modelo no teste.

In [ ]:
# O melhor modelo encontrado, já treinado, avaliado no conjunto de teste
modelo_otimizado = grid.best_estimator_
y_pred_otim = avaliar_modelo("Regressão Logística (otimizada)", modelo_otimizado)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_otim, display_labels=["Ficou", "Cancelou"], cmap="Blues"
)
plt.title("Matriz de confusão — Regressão Logística otimizada")
plt.show()

**Leitura — a otimização revelou um trade-off importante:**

| Métrica | Log. Regressão (padrão) | Log. Regressão (otimizada) | Efeito |
|---|---|---|---|
| Acurácia | 0,809 | 0,743 | caiu |
| Precisão | 0,667 | 0,511 | caiu |
| **Recall** | 0,560 | **0,790** | **subiu muito** |
| F1 | 0,609 | 0,620 | subiu um pouco |

O `class_weight="balanced"` fez o modelo dar **mais atenção à classe minoritária** (os que cancelam). O resultado: o **recall saltou de 56% para 79%** — agora o modelo detecta quase 4 em cada 5 clientes que vão cancelar (antes pegava pouco mais da metade). O preço foi a **queda na precisão** (mais alarmes falsos) e na acurácia.

Isso é uma melhora ou uma piora? Depende do objetivo — e aqui é onde a análise vira decisão de negócio. Para **retenção de clientes**, esse trade-off geralmente compensa: é melhor abordar alguns clientes que não iam sair (falso positivo, custo baixo — uma ligação, um desconto) do que deixar escapar quem ia cancelar (falso negativo, custo alto — receita perdida). Como priorizamos o **recall/F1**, e ambos melhoraram, o modelo otimizado é a melhor solução para o nosso objetivo, apesar da acurácia menor.

## 8. Avaliação final dos resultados

Aqui aprofundamos a análise: verificamos **overfitting/underfitting**, analisamos os **erros** do modelo escolhido e discutimos **limitações e melhorias**.

### 8.1 Verificação de overfitting / underfitting

Um modelo que vai **muito melhor no treino do que no teste** está "decorando" em vez de aprender (**overfitting**). Comparamos o F1 nos dois conjuntos: quanto maior a diferença, maior o overfitting.

In [ ]:
from sklearn.metrics import f1_score

# Comparar F1 no TREINO vs no TESTE -> diferença grande indica overfitting
modelos_treinados = {
    "Regressão Logística": pipe_lr,
    "Random Forest": pipe_rf,
    "Log. Regressão (otimizada)": modelo_otimizado,
}

linhas = []
for nome, modelo in modelos_treinados.items():
    f1_treino = f1_score(y_train, modelo.predict(X_train))
    f1_teste = f1_score(y_test, modelo.predict(X_test))
    linhas.append({
        "Modelo": nome,
        "F1 treino": round(f1_treino, 3),
        "F1 teste": round(f1_teste, 3),
        "Diferença": round(f1_treino - f1_teste, 3),
    })

pd.DataFrame(linhas).set_index("Modelo")

**Leitura — diagnóstico de overfitting:**
- **Regressão Logística** (padrão e otimizada): desempenho **quase igual** no treino e no teste → os modelos **generalizam bem**, aprenderam o *padrão* e não os exemplos.
- **Random Forest**: F1 de **~1,00 no treino** contra **~0,53 no teste** — um abismo. É um caso claro de **overfitting**: o modelo praticamente *decorou* os dados de treino, mas não sabe lidar com clientes novos. Isso explica por que ela ficou atrás na comparação. Poderia ser atenuada limitando a profundidade das árvores, mas, como a Regressão Logística já resolve bem e é mais simples/interpretável, seguimos com ela.
- Nenhum modelo apresentou **underfitting** grave (o baseline, sim, "sub-ajusta" por design — não aprende nada).

### 8.2 Análise de erros do modelo escolhido

O **modelo escolhido** é a **Regressão Logística otimizada**, por ter o melhor equilíbrio para o objetivo (maior recall e melhor F1). Olhando a matriz de confusão dele no teste (2.113 clientes):

- **443 dos 561** clientes que realmente cancelaram foram **identificados** (recall 79%).
- **118 canceladores escaparam** (falsos negativos) — o modelo disse que ficariam, mas cancelaram. É o erro mais custoso para o negócio.
- **424 falsos positivos** — clientes sinalizados como risco que na verdade ficariam. Custo baixo: no máximo uma ação de retenção "desnecessária".

**Interpretação:** o modelo erra mais "para o lado seguro" (prefere sinalizar demais a deixar escapar), o que é adequado para retenção. A empresa pode ajustar esse equilíbrio conforme o orçamento disponível para campanhas.

### 8.3 Limitações e melhorias futuras

**Limitações da solução:**
- O **recall de ~79%** é bom, mas ainda **deixa ~21% dos cancelamentos escaparem** — não é uma solução perfeita.
- O aumento do recall veio com **queda de precisão** (mais alarmes falsos): a empresa gastaria retenção com alguns clientes que não iam sair.
- O dataset é **fictício** e estático (sem histórico temporal), então os padrões podem ser mais "limpos" do que na realidade.
- Não temos o **valor financeiro** de cada cliente, então otimizamos por F1 — e não pelo **lucro real** da campanha de retenção.

**Possíveis melhorias futuras:**
- Testar modelos de **boosting** (Gradient Boosting, XGBoost), que costumam ir bem em dados tabulares.
- **Calibrar o limiar de decisão** (threshold) para ajustar o equilíbrio recall × precisão conforme o orçamento da empresa.
- **Engenharia de atributos** (ex.: combinar serviços contratados, criar faixas de `tenure`).
- Incorporar **novos dados** (reclamações, satisfação, uso do serviço).
- Otimizar diretamente por uma **métrica de negócio** (retorno financeiro), não só estatística.

## 9. Conclusão

**O problema.** O objetivo deste MVP foi prever o **cancelamento de clientes (churn)** de uma operadora de telecom — um problema de **classificação binária** com forte valor de negócio, já que reter um cliente costuma custar muito menos do que conquistar um novo.

**Os dados.** Usamos o dataset público *Telco Customer Churn* (IBM), com **7.043 clientes** e 20 atributos preditores. A variável-alvo (`Churn`) é **desbalanceada**: apenas 26,5% dos clientes cancelaram.

**Os tratamentos.** Corrigimos a coluna `TotalCharges` (que veio como texto, com 11 valores em branco de clientes recém-chegados, preenchidos com 0), removemos o identificador `customerID`, e montamos um **pipeline de pré-processamento** (padronização das numéricas + codificação one-hot das categóricas) ajustado **apenas no treino**, para evitar **vazamento de dados**. A divisão foi **70/30 estratificada**.

**Os modelos avaliados.** Comparamos um **baseline** (sempre prever "não cancela"), uma **Regressão Logística** e uma **Random Forest**; depois **otimizamos** a Regressão Logística com Grid Search + validação cruzada.

**O melhor resultado.** A **Regressão Logística otimizada** (`C=0.01`, `class_weight="balanced"`) foi a escolhida: ela alcançou **recall de 79%** e o **melhor F1 (0,62)**. Foi preferida porque, no contexto de retenção, **detectar o máximo de clientes em risco (recall alto) é mais valioso** do que maximizar a acurácia — e ela faz isso sem sinais de overfitting (desempenho semelhante no treino e no teste), além de ser simples e interpretável.

**Por que essa e não a Random Forest?** Apesar de mais complexa, a Random Forest apresentou **overfitting severo** (quase perfeita no treino, fraca no teste) e ficou atrás nas métricas de interesse.

**Cumprimos o objetivo?** Sim. Saímos de um baseline **inútil** (recall 0) para um modelo que identifica **~4 de cada 5 clientes** prestes a cancelar — uma ferramenta que permitiria à empresa **agir antes** e reduzir perdas.

**Próximos passos.** Testar modelos de boosting, calibrar o limiar de decisão conforme o orçamento de retenção, fazer engenharia de atributos e, idealmente, otimizar por uma métrica de **retorno financeiro** em vez de apenas estatística.

> Do problema inicial à solução final, o notebook mostrou o **fluxo completo** de um projeto de Machine Learning — com cada decisão justificada.

## 10. Boas práticas e checklist do MVP

Esta seção registra as boas práticas adotadas e responde, de forma objetiva, às perguntas do checklist sugerido pelo MVP.

### 10.1 Bibliotecas utilizadas e reprodutibilidade

**Bibliotecas:** `pandas` e `numpy` (manipulação de dados), `matplotlib` e `seaborn` (visualização) e `scikit-learn` (pré-processamento, modelos, métricas e otimização). Todas já vêm instaladas no Google Colab — não é preciso instalar nada.

**Reprodutibilidade:** fixamos uma semente (`SEED = 42`) usada na divisão treino/teste, na validação cruzada e nos modelos, garantindo que qualquer execução produza os mesmos resultados. Os dados são lidos de uma **URL pública**, então o notebook roda do início ao fim sem configuração manual.

**Recursos:** o treinamento é leve (segundos em CPU comum) — não exige GPU nem hardware especial.

### 10.2 Respostas ao checklist do MVP

**Definição do problema**
- *Descrição do problema:* prever o cancelamento (churn) de clientes de uma operadora de telecom.
- *Objetivo do modelo:* identificar clientes com alto risco de cancelar, para permitir ações de retenção.
- *Tipo de problema:* classificação binária supervisionada.
- *Por que ML:* há exemplos históricos rotulados (cancelou/ficou); o modelo aprende o padrão e prevê novos casos.
- *Premissas/hipóteses:* dados históricos representam o comportamento futuro; contrato, tempo de casa e forma de pagamento influenciam o churn.
- *Restrições:* usamos apenas os atributos do dataset; base pública e não utilizada nas aulas.

**Descrição dos dados**
- *Dataset e fonte:* Telco Customer Churn (IBM, base pública e fictícia).
- *Como foi carregado:* diretamente de uma URL `raw` pública do GitHub, sem upload/login.
- *Registros e atributos:* 7.043 clientes × 21 colunas (20 preditores + alvo).
- *Principais atributos:* `Contract`, `tenure`, `PaymentMethod`, `InternetService`, `MonthlyCharges`, `TotalCharges`.
- *Variável-alvo:* `Churn` (Yes/No → 1/0).
- *Limitações:* dados fictícios, estáticos, sem histórico de uso ou satisfação.

**Preparação e divisão**
- *Valores ausentes:* 11 em `TotalCharges` (clientes com `tenure=0`) → preenchidos com 0 (regra de negócio).
- *Remoção/transformação:* `customerID` removido; alvo convertido para 0/1.
- *Codificação/normalização:* one-hot nas categóricas + padronização nas numéricas, dentro de um pipeline.
- *Vazamento de dados:* evitado — pré-processamento ajustado **só no treino**.
- *Divisão:* 70% treino / 30% teste, **estratificada**; validação cruzada usada na otimização.

**Modelagem, otimização e avaliação**
- *Baseline:* `DummyClassifier` (sempre "não cancela") — acurácia 0,735, recall 0.
- *Modelos treinados:* Regressão Logística e Random Forest (ambos além do baseline).
- *Comparação justa:* mesmo pipeline e mesmo conjunto de teste para todos.
- *Underfitting/overfitting:* Random Forest apresentou **overfitting** (F1 treino ~1,0 × teste ~0,53); a Regressão Logística generalizou bem.
- *Otimização:* Grid Search + validação cruzada (5 dobras) ajustando `C` e `class_weight` da Regressão Logística; recall subiu de 0,56 para 0,79.
- *Métricas e por quê:* acurácia, precisão, recall, F1 e matriz de confusão — com foco em **recall/F1** por causa do desbalanceamento.
- *Melhor modelo:* Regressão Logística otimizada (melhor recall e F1, sem overfitting, interpretável).

**Conclusão**
- *Melhor solução e por quê:* Regressão Logística otimizada — melhor equilíbrio para o objetivo de retenção.
- *Cumpriu o objetivo?* Sim: de recall 0 (baseline) para 79%.
- *Próximos passos:* boosting, calibração de limiar, engenharia de atributos, otimização por retorno financeiro.